In [ ]:
import pandas as pd
from pathlib import Path

from tgbs_rs.config.config import TABLES_DIR, HLS_MERGED_BANDS, S2_ALL_BANDS
from tgbs_rs.visualization import (
    prepare_composite_table,
    merge_monthly_to_full_grid,
    add_period_label,
    add_season_label,
    add_valid_value_flag,
    flag_rows_with_missing_values,
    sort_site_time,
    aggregate_monthly_to_seasonal_with_support
)

#### Read raw Data Tables

In [2]:
csv_paths = {
    "hls_annual": TABLES_DIR / "tgbs_hls_annual_composite_all_bands_2014_2025_full_raw.csv",
    "hls_monthly": TABLES_DIR / "tgbs_hls_monthly_composite_all_bands_2014_2025_full_raw.csv",
    "s2_annual": TABLES_DIR / "tgbs_s2_annual_composite_all_bands_2019_2025_full_raw.csv",
    "s2_monthly": TABLES_DIR / "tgbs_s2_monthly_composite_all_bands_2019_2025_full_raw.csv",
}

hls_cols = HLS_MERGED_BANDS
s2_cols = S2_ALL_BANDS

In [3]:
hls_annual_df = pd.read_csv(csv_paths["hls_annual"])
hls_monthly_df = pd.read_csv(csv_paths["hls_monthly"])
s2_annual_df = pd.read_csv(csv_paths["s2_annual"])
s2_monthly_df = pd.read_csv(csv_paths["s2_monthly"])

hls_annual_df = hls_annual_df.drop(columns=[".geo"])
s2_annual_df = s2_annual_df.drop(columns=[".geo"])
hls_monthly_df = hls_monthly_df.drop(columns=[".geo"])
s2_monthly_df = s2_monthly_df.drop(columns=[".geo"])

#### Prepare Annual Tables
    - remove structurally irrelevant temporal columns
    - add baseline/current period labels
    - add missing-value flags
    - sort for reproducibility

In [89]:
hls_annual_prepped = prepare_composite_table(
    hls_annual_df,
    temporal_scale="annual",
    value_columns=hls_cols,
    drop_incomplete_rows=False,
)

In [90]:
s2_annual_prepped = prepare_composite_table(
    df=s2_annual_df,
    temporal_scale="annual",
    value_columns=s2_cols,
    drop_incomplete_rows=False,
)

In [91]:
hls_annual_prepped.to_csv(TABLES_DIR / "tgbs_hls_annual_master_all_bands_2014_2025.csv")

In [92]:
s2_annual_prepped.to_csv(TABLES_DIR / "tgbs_s2_annual_master_all_bands_2019_2025.csv")

##### Prepare Monthly Tables
    - remove structurally irrelevant temporal columns
    - add baseline/current period labels
    - add missing-value flags
    - sort for reproducibility

In [4]:
hls_monthly_prepped = prepare_composite_table(
    df=hls_monthly_df,
    temporal_scale="monthly",
    value_columns=hls_cols,
    drop_incomplete_rows=False,
)

In [5]:
s2_monthly_prepped = prepare_composite_table(
    df=s2_monthly_df,
    temporal_scale="monthly",
    value_columns=s2_cols,
    drop_incomplete_rows=False,
)

##### Expand HLS monthly table onto the full expected site-year-month grid
    - makes missing site-month rows explicit

In [6]:
hls_monthly_prepped = merge_monthly_to_full_grid(
    monthly_df=hls_monthly_prepped,
    start_year=2014,
    end_year=2025,
)

In [7]:
s2_monthly_prepped = merge_monthly_to_full_grid(
    monthly_df=s2_monthly_prepped,
    start_year=2019,
    end_year=2025,
)

##### Re-add period and season labels after grid completion
    - period is useful for baseline/current summaries
    - season is needed for wet/dry aggregation

In [8]:
hls_monthly_prepped = add_period_label(
    df=hls_monthly_prepped,
    baseline_start=2014,
    baseline_end=2017,
    current_start=2018,
    current_end=2025,
)

hls_monthly_prepped = add_season_label(
    df=hls_monthly_prepped,
    wet_months=[3, 4, 5],
    dry_months=[7, 8, 9, 10],
)

hls_monthly_prepped = add_valid_value_flag(
    df=hls_monthly_prepped,
    value_columns=hls_cols,
)

hls_monthly_prepped = flag_rows_with_missing_values(
    df=hls_monthly_prepped,
    value_columns=hls_cols,
)

hls_monthly_prepped = sort_site_time(hls_monthly_prepped)

In [9]:
s2_monthly_prepped = add_period_label(
    df=s2_monthly_prepped,
    baseline_start=2014,
    baseline_end=2017,
    current_start=2018,
    current_end=2025,
)

s2_monthly_prepped = add_season_label(
    df=s2_monthly_prepped,
    wet_months=[3, 4, 5],
    dry_months=[7, 8, 9, 10],
)

s2_monthly_prepped = add_valid_value_flag(
    df=s2_monthly_prepped,
    value_columns=s2_cols,
)

s2_monthly_prepped = flag_rows_with_missing_values(
    df=s2_monthly_prepped,
    value_columns=s2_cols,
)

s2_monthly_prepped = sort_site_time(s2_monthly_prepped)


In [99]:
hls_monthly_prepped.to_csv(TABLES_DIR / "tgbs_hls_monthly_master_all_bands_2014_2025.csv")

In [100]:
s2_monthly_prepped.to_csv(TABLES_DIR / "tgbs_s2_monthly_master_all_bands_2019_2025.csv")

##### Generate Seasonal Tables

In [10]:
hls_seasonal_df = aggregate_monthly_to_seasonal_with_support(
    df=hls_monthly_prepped,
    value_columns=hls_cols,
    min_valid_months_by_season={"wet": 2, "dry": 3},
)

In [11]:
s2_seasonal_df = aggregate_monthly_to_seasonal_with_support(
    df=s2_monthly_prepped,
    value_columns=s2_cols,
    min_valid_months_by_season={"wet": 2, "dry": 3},
)

In [13]:
hls_seasonal_df.to_csv(TABLES_DIR / "tgbs_hls_seasonal_master_all_bands_2014_2025.csv")

In [ ]:
s2_seasonal_df.to_csv(TABLES_DIR / "tgbs_s2_seasonal_master_all_bands_2019_2025.csv")